# Python 101 - Solutions
## Chapter XII

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_12.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

BASE = './data/'

df = pd.read_excel(io=BASE + 'currency_rates.xlsx', sheet_name='data')
df = df.set_index('Date').sort_index()

currencies = ['EUR/USD', 'GBP/USD', 'EUR/GBP']


def curr_list(suffix):
    return [currency + suffix for currency in currencies]


print('range:', df.index.min().date(), '-', df.index.max().date(), f'({len(df)} rows)')
df.head()

### 1. EUR/USD around Trump's election (2016-11-08)

Worth saying out loud: the file **ends on 2016-11-11**, three days after the election. So you can see the jump, but not the aftermath. Noticing the limits of your data before drawing conclusions is the actual lesson here.

In [ ]:
window = df.loc['2016-10-15':, 'EUR/USD Close']
window.plot(figsize=(11, 5), title="EUR/USD around the 2016 US election")
plt.axvline(pd.Timestamp('2016-11-08'), color='red', linestyle='--', label='election day')
plt.legend();

before = df.loc['2016-11-07', 'EUR/USD Close']
after = df.loc['2016-11-09', 'EUR/USD Close']
print(f'7 Nov: {before:.4f}  ->  9 Nov: {after:.4f}  ({(after / before - 1):+.2%})')

assert df.index.max() == pd.Timestamp('2016-11-11')
assert after < before                                 # the dollar strengthened

### 2. Which currency 'won' the election?

`EUR/USD` and `EUR/GBP` both fell while `GBP/USD` rose - so the **dollar** gained against the euro, and the **pound** was the only pair up on the day. Which one 'won' depends on which side of the slash you are standing on, and that is worth ten seconds of discussion.

In [ ]:
before = df.loc['2016-11-07', curr_list(' Close')]
after = df.loc['2016-11-09', curr_list(' Close')]

change = ((after.values / before.values) - 1) * 100
summary = pd.DataFrame({'before': before.values, 'after': after.values,
                        'change_%': change}, index=currencies)

print(summary.round(4).to_string())
print(f"\nbiggest move: {summary['change_%'].abs().idxmax()}")
print(f"only pair up: {summary.loc[summary['change_%'] > 0].index.tolist()}")

assert summary.loc['GBP/USD', 'change_%'] > 0
assert summary.loc['EUR/USD', 'change_%'] < 0
assert summary['change_%'].abs().idxmax() == 'EUR/GBP'

### 3. 30-day moving sum of the *up* days

The three steps from the notebook's comments: `diff()` to get the daily move, `where` to keep only the positive ones, then `rolling(30).sum()`.

`where(cond, 0)` keeps the value where the condition holds and puts `0` everywhere else - that is what makes the rolling window still cover 30 calendar rows rather than 30 up-days.

In [ ]:
daily_change = df['EUR/USD Close'].diff()
gains_only = daily_change.where(daily_change > 0, 0)
moving_gain = gains_only.rolling(window=30).sum()

moving_gain.plot(figsize=(11, 5),
                 title='EUR/USD: sum of the up-moves over a 30 day window');

print(moving_gain.dropna().tail(3).round(4).to_string())

assert moving_gain.isna().sum() == 29          # 30-day window needs 30 rows
assert (moving_gain.dropna() >= 0).all()
assert (gains_only < 0).sum() == 0

### 4.a 60-day moving standard deviation

$${stddev} = \sqrt{\frac{\sum{({x}-\bar{x})^2}}{N}}$$

One thing to flag: `rolling().std()` uses the **sample** standard deviation (dividing by `N-1`) by default, while the formula above divides by `N`. Pass `ddof=0` if you want it to match the formula exactly. The difference is tiny here, but 'the library and the formula disagree slightly' is a good thing for them to have met once.

In [ ]:
rolling_std = df['EUR/USD Close'].rolling(window=60).std()
rolling_std_population = df['EUR/USD Close'].rolling(window=60).std(ddof=0)

print(rolling_std.dropna().tail(3).round(5).to_string())
print()
print('sample (ddof=1, the default):', round(rolling_std.dropna().iloc[-1], 6))
print('population (ddof=0, formula):', round(rolling_std_population.dropna().iloc[-1], 6))

rolling_std.plot(figsize=(11, 5), title='EUR/USD: 60 day rolling volatility');

assert rolling_std.isna().sum() == 59
assert rolling_std.dropna().iloc[-1] > rolling_std_population.dropna().iloc[-1]

### 4.b The original and the quarterly average together

`resample('QS')` buckets by quarter-start, then `.mean()` collapses each bucket. Plotting both on the same axes shows what the resampling threw away.

In [ ]:
closing = df['EUR/USD Close']
quarterly = closing.resample('QS').mean()

ax = closing.plot(figsize=(11, 5), label='EUR/USD Close')
quarterly.plot(ax=ax, style='--g', label='quarterly mean')
plt.title('EUR/USD: daily closes and the quarterly average')
plt.legend();

print(quarterly.round(4).to_string())

assert len(quarterly) == 5              # Q4 2015 .. Q4 2016
assert closing.min() < quarterly.min() and quarterly.max() < closing.max()